# Confidence vs Cytescore analysis
## Prep

In [ ]:
import json
import os
from pathlib import Path
import random

import pandas as pd
import scanpy as sc

from storage import download_from_r2, fetch_uploaded_r2_keys
from shared.repo import REPO_ROOT

In [ ]:
r2_keys_set = fetch_uploaded_r2_keys()
r2_keys = list(r2_keys_set)
unique_prefixes = set()
for key in r2_keys:
    prefix = key.split("/")[0] if "/" in key else key
    if prefix not in unique_prefixes:
        unique_prefixes.add(prefix)
list(unique_prefixes)

In [ ]:
annotated_h5ad_keys = [key for key in r2_keys if key.startswith("cytetype_pipeline_20260522_175813")]

## Download and read `h5ad`

### Download from r2

In [ ]:
seed = 15
random.seed(seed)
i = random.randint(0, len(annotated_h5ad_keys) - 1)
ACCESSION = annotated_h5ad_keys[i].split("/")[-1].split("_")[0]
LOCAL_PATH = REPO_ROOT / "tmp" / f"{annotated_h5ad_keys[i].split('/')[-1]}"
download_from_r2(annotated_h5ad_keys[i], LOCAL_PATH)
print(f"File {i}: {ACCESSION}")

### Read downloaded file

In [ ]:
adata = sc.read(LOCAL_PATH)
adata.uns["cytetype_jobDetails"]["report_url"]

## Load locally cached `anndata` object

In [ ]:
LOCAL_H5AD_FOLDER = REPO_ROOT / "data/cytetype_annotated"

# show available files
available_files = os.listdir(LOCAL_H5AD_FOLDER)
available_files

In [ ]:
H5AD_FILE_INDEX = 0

# import h5ad file with STATE and CyteType labels
ACCESSION = available_files[H5AD_FILE_INDEX].split("_")[0]
adata = sc.read(LOCAL_H5AD_FOLDER / available_files[H5AD_FILE_INDEX])

In [ ]:
len(adata.obs)-95

## Merge CyteScore and CyteType confidence into `.obs` DataFrame
### CyteScore

In [ ]:
# import CyteScores, filter by accession
df = pd.read_csv(REPO_ROOT / f"output/cyteonto_pipeline/deduplicated_tables/deduplicated.csv")
df = df[df["accession"] == ACCESSION]

# create key column in adata
adata.obs["pair_label"] = adata.obs["cell_type"].astype(str) + adata.obs["cytetype_annotation_leiden_merged"].astype(str)

# set keys as indices in the two dfs to be joined
df = df.set_index("pair_label")
adata.obs = adata.obs.set_index("pair_label")

# join on STATE-CyteType key
adata.obs = adata.obs.join(df)

### CyteType confidence

In [ ]:
CLUSTER_KEY = "leiden_merged"

payload = adata.uns["cytetype_results"]["result"]
cytetype_result = json.loads(payload) if isinstance(payload, str) else payload

confidence_by_cluster = {
    str(cluster_id): entry["latest"]["review"]["confidence"]
    for cluster_id, entry in cytetype_result["raw_annotations"].items()
}

adata.obs["cytetype_confidence"] = (
    adata.obs[CLUSTER_KEY]
    .map(confidence_by_cluster)
)

### UMAP colored by cluster confidence

In [ ]:
CONFIDENCE_ORDER = ["High", "Moderate", "Low"]
N_EXTREMES = 10

pairs = (
    adata.obs.reset_index()[
        ["cell_type", "cytetype_annotation_leiden_merged", "cytescore_similarity", "cytetype_confidence"]
    ]
    .drop_duplicates(["cell_type", "cytetype_annotation_leiden_merged"])
    .dropna(subset=["cytescore_similarity"])
)

cytetype_order = (
    pairs.groupby("cytetype_annotation_leiden_merged", observed=True)["cytetype_confidence"]
    .first()
    .reset_index()
    .assign(
        cytetype_confidence=lambda d: pd.Categorical(
            d["cytetype_confidence"], categories=CONFIDENCE_ORDER, ordered=True
        )
    )
    .sort_values(["cytetype_confidence", "cytetype_annotation_leiden_merged"])
)

rows = []
for _, meta in cytetype_order.iterrows():
    cytetype = meta["cytetype_annotation_leiden_merged"]
    conf = meta["cytetype_confidence"]
    sub = pairs[pairs["cytetype_annotation_leiden_merged"] == cytetype]
    n_take = min(N_EXTREMES, len(sub))
    top = sub.nlargest(n_take, "cytescore_similarity")
    bottom = sub.nsmallest(n_take, "cytescore_similarity")
    n_rows = max(len(top), len(bottom))
    for rank in range(n_rows):
        rows.append(
            {
                "cytetype_annotation_leiden_merged": cytetype if rank == 0 else "",
                "cytetype_confidence": conf if rank == 0 else "",
                "rank": rank + 1,
                "top_cell_type": top.iloc[rank]["cell_type"] if rank < len(top) else pd.NA,
                "top_cytescore_similarity": top.iloc[rank]["cytescore_similarity"] if rank < len(top) else pd.NA,
                "bottom_cell_type": bottom.iloc[rank]["cell_type"] if rank < len(bottom) else pd.NA,
                "bottom_cytescore_similarity": bottom.iloc[rank]["cytescore_similarity"] if rank < len(bottom) else pd.NA,
            }
        )

extremes_by_cytetype = pd.DataFrame(rows)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(extremes_by_cytetype)

In [ ]:
extremes_by_cytetype[extremes_by_cytetype["cytetype_confidence"] == "Low"]["bottom_cell_type"].value_counts()

In [ ]:
sc.pl.umap(adata, color=["cytescore_similarity", "cytetype_confidence"], ncols=2)

In [ ]:
from scanpy.plotting._utils import set_colors_for_categorical_obs

print(adata.uns["cytetype_jobDetails"]["report_url"])

confidence_palette = {"Low": "#d73027", "Moderate": "#fee08b", "High": "#1a9850"}
set_colors_for_categorical_obs(adata, "cytetype_confidence", confidence_palette)

sc.pl.umap(
    adata,
    color=[
        "cell_type",
        "cytetype_annotation_leiden_merged",
        "cytescore_similarity",
        "cytetype_confidence",
    ],
    title=[
        f"{ACCESSION}: STATE labels",
        f"{ACCESSION}: CyteType annotations",
        f"{ACCESSION}: CyteOnto cytescore similarity",
        f"{ACCESSION}: CyteType cluster confidence",
    ],
    ncols=2,
    legend_loc="right margin",
    size=30,
    wspace=0.4,
)

# Clean up `h5ad`

In [ ]:
# os.remove(LOCAL_PATH)